# STA 141B Final Project
## Geographical Analysis of Ticketmaster Events in the United States
### Peyton Lee, Joy Liu, Katriel Layam, Leah Aviles

In [ ]:
# import packages
import requests
import json
import pandas as pd
import time
import folium
import sqlite3

from datetime import datetime

To access the ticketmaster API, each person is assigned a unique API consumer key. We each created txt files that have our API keys and stored it
in a filepath string variable for easier and cleaner access to our key.

In [ ]:
# PEYTON RUN
filepath = "/Users/peyton/Desktop/TicketmasterAPIKey.txt"

In [ ]:
# KATRIEL RUN
filepath = "/Users/katriellayam/Downloads/ticketmaster_project.txt"

In [ ]:
# LEAH RUN
filepath = "/Users/leahaviles/Downloads/ticket_masterAPIKey.txt"

In [ ]:
# JOY RUN
filepath = "/Users/joy/Downloads/ticketmaster api key.txt"

In [ ]:
# function read_key that reads in API key txt file and returns stripped version of key
def read_key(keyfile):
    with open(keyfile) as f:
        return f.readline().strip("\n")

# use read_key function on our filepath and set it as key variable to easily access API key
key = read_key(filepath)

# check type of the key
type(key)

Create Dataframe of all ticketmaster events from March 2026 to February 2027

This Dataframe will include the following information for each event:
* Name
* Date
* Venue
* City
* State
* Latitude
* Longitude
* URL
* Month
* Segment
* Genre
* Subgenre

In [ ]:
# RUN ONCE

all_events = [] # create empty list to store all event information

# create helper function with inputs of start date and end date of event and a label of the date (if applicable)
def fetch_events(start, end, label):
    """
    Function that fetches events between a specified start and end date
    """
    page = 0 # set starting page to 0 since Ticketmaster API results are returned in pages
    chunk_events = [] # create empty list that stores event information in the specified time chunk

    # keep requesting pages while page number is less than 5
    # this is done as a safety limit
    while page < 5:
        # create API request url
        url = (
            f"https://app.ticketmaster.com/discovery/v2/events.json" # ticketmaster events endpoint url
            f"?apikey={key}" # pass API key
            f"&startDateTime={start}T00:00:00Z" # events starting from the beginning of the start date
            f"&endDateTime={end}T23:59:59Z" # events up to the end of the end date
            f"&countryCode=US" # only include events in the United States
            f"&size=200" # request up to 200 events per page
            f"&page={page}" # get page number the loop is on
        )

        # request response to get data from our specified url
        response = requests.get(url)

        # if there is a bad request, print message and stop the chunk
        if response.status_code == 400:
            print(f"  400 error on page {page+1}, stopping this chunk.")
            break

        # check status of requested response, raises an error for failed requests
        response.raise_for_status()

        # convert data results from a JSON to a Python dictionary structure
        data = response.json()

        # extract the events from the data results
        events = data.get("_embedded", {}).get("events", [])

        # if events section of data is empty, stop the page loop
        if not events:
            break

        # loop through each event that is in the page
        for e in events:
            # try and except block for processing each event
            try:
                venue = e.get("_embedded", {}).get("venues", [{}])[0] # extract venue information for the event
                loc = venue.get("location", {}) # get the venue's location

                # extract classifications information
                classifications = e.get("classifications", [{}])
                # if there is a list of classifications, extract it
                primary = classifications[0] if classifications else {}

                # extract each category in the classifications data for each event, if it is missing, return {}
                # then get the name for each category of the event
                segment  = primary.get("segment", {}).get("name")
                genre    = primary.get("genre", {}).get("name")
                subgenre = primary.get("subGenre", {}).get("name")

                # append event data to the empty chunk_events list
                chunk_events.append({
                    "name":     e.get("name"),
                    "date":     e.get("dates", {}).get("start", {}).get("localDate"),
                    "venue":    venue.get("name"),
                    "city":     venue.get("city", {}).get("name"),
                    "state":    venue.get("state", {}).get("stateCode"),
                    "latitude":  loc.get("latitude"),
                    "longitude": loc.get("longitude"),
                    "url":      e.get("url"),
                    "month":    start[:7],
                    "segment":  segment,
                    "genre":    genre,
                    "subgenre": subgenre,
                })
            # if there is an error when processing an event, print error and then skip that specific event
            except Exception as ex:
                print(f" Skipping event: {ex}")
                continue

        # get page data from API resposne
        page_info = data.get("page", {})

        # get total number of pages
        total_pages = page_info.get("totalPages", 1)

        # print out progression of the processing of events
        print(f"{start[:7]} — page {page+1}/{total_pages} — {len(chunk_events)} events so far")

        # if the next page goes past total number of pages, stop loop
        if page + 1 >= total_pages:
            break

        # go to next page
        page += 1

        # wait for 0.25 seconds between each request
        time.sleep(0.25)

    return chunk_events # return list of events from the date chunk

# define the date ranges in a list (March 2026 to February 2027
# split into 4 weekly chunks per month to reduce hitting page limits
month_chunks = [
    # March 2026
    ("2026-03-01", "2026-03-07"), ("2026-03-08", "2026-03-15"),
    ("2026-03-16", "2026-03-23"), ("2026-03-24", "2026-03-31"),
    # April 2026
    ("2026-04-01", "2026-04-07"), ("2026-04-08", "2026-04-15"),
    ("2026-04-16", "2026-04-23"), ("2026-04-24", "2026-04-30"),
    # May 2026
    ("2026-05-01", "2026-05-07"), ("2026-05-08", "2026-05-15"),
    ("2026-05-16", "2026-05-23"), ("2026-05-24", "2026-05-31"),
    # June 2026
    ("2026-06-01", "2026-06-07"), ("2026-06-08", "2026-06-15"),
    ("2026-06-16", "2026-06-23"), ("2026-06-24", "2026-06-30"),
    # July 2026
    ("2026-07-01", "2026-07-07"), ("2026-07-08", "2026-07-15"),
    ("2026-07-16", "2026-07-23"), ("2026-07-24", "2026-07-31"),
    # August 2026
    ("2026-08-01", "2026-08-07"), ("2026-08-08", "2026-08-15"),
    ("2026-08-16", "2026-08-23"), ("2026-08-24", "2026-08-31"),
    # September 2026
    ("2026-09-01", "2026-09-07"), ("2026-09-08", "2026-09-15"),
    ("2026-09-16", "2026-09-23"), ("2026-09-24", "2026-09-30"),
    # October 2026
    ("2026-10-01", "2026-10-07"), ("2026-10-08", "2026-10-15"),
    ("2026-10-16", "2026-10-23"), ("2026-10-24", "2026-10-31"),
    # November 2026
    ("2026-11-01", "2026-11-07"), ("2026-11-08", "2026-11-15"),
    ("2026-11-16", "2026-11-23"), ("2026-11-24", "2026-11-30"),
    # December 2026
    ("2026-12-01", "2026-12-07"), ("2026-12-08", "2026-12-15"),
    ("2026-12-16", "2026-12-23"), ("2026-12-24", "2026-12-31"),
    # January 2027
    ("2027-01-01", "2027-01-07"), ("2027-01-08", "2027-01-15"),
    ("2027-01-16", "2027-01-23"), ("2027-01-24", "2027-01-31"),
    # February 2027
    ("2027-02-01", "2027-02-07"), ("2027-02-08", "2027-02-15"),
    ("2027-02-16", "2027-02-23"), ("2027-02-24", "2027-02-28"),
]

# loop through every weekly chunk in month_chunks
for start, end in month_chunks:
    label = start[:7] # extract year and month from the start date
    events = fetch_events(start, end, label) # call fetch_events function for each chunk and store data in events
    all_events.extend(events) # add all returned events into the all_events list created earlier
    print(f"Running total: {len(all_events)} events \n") # print the current total number of events
    time.sleep(0.25) # wait for 0.25 seconds when processing between chunks

# convert all_events into a pandas DataFrame
events_df = pd.DataFrame(all_events)
events_df = events_df.dropna(subset=["latitude", "longitude"]) # remove rows wehre latitude or longitude is missing

# convert latitude and longitude values into floating point numbers
events_df["latitude"]  = events_df["latitude"].astype(float)
events_df["longitude"] = events_df["longitude"].astype(float)

# remove repeated rows where event name, date, and venue are the same
# this keeps only one copy of each event and avoids the same event from appearing more than once
events_df = events_df.drop_duplicates(subset=["name", "date", "venue"])

# print the total number of events in the dataframe
print(f"\nTotal events: {len(events_df)}")

# print out the most common classifications
print(events_df[["segment", "genre", "subgenre"]].value_counts().head(20)) # top 20

# store dataframe in a CSV
output_file = "classified_ticketmaster_events.csv" # name of output file
events_df.to_csv(output_file, index=False) # save dataframe to a CSV file

# make sure dataframe has been saved to CSV file and show length of dataframe
print(f"Saved {len(events_df)} events to {output_file}")

Now that our desired data has been saved to a CSV, we perform exploratory data analysis (EDA)

In [ ]:
#  RUN ONCE
# read CSV file and save as events_df
events_df = pd.read_csv("classified_ticketmaster_events.csv")

# explore data
events_df.head() # first 5 rows of dataframe
print(events_df.shape) # check number of rows and columns
print(events_df.head()) # print out first 5 rows

print(events_df.info()) # print information about structure of dataframe

print(events_df.isna().sum()) # count missing values in each column

In [ ]:
#  RUN ONCE
# closer exploration of rows

# inspect rows with missing values
events_df[events_df["state"].isna()].head(20) # rows where state is missing
events_df[events_df["city"].isna()].head(20) # rows where city is missing
events_df[events_df["segment"].isna()].head(20) # rows where segment is missing
events_df[events_df["genre"].isna()].head(20) # rows where genre is missing
events_df[events_df["subgenre"].isna()].head(20) # rows where subgenre is missing

# check for unique state values
# confirms we have all 50 US States
sorted(events_df["state"].dropna().unique())

events_df["segment"].value_counts() # count occurrences of each segment category

events_df["genre"].value_counts().head(20) # count occurrences for each genre category (top 20)

events_df["subgenre"].value_counts().head(20) # count occurrences for each subgenre category (top 20)

# show top 20 states that have the most events
events_df["state"].value_counts().head(20)

# show the top 20 cities that have the most events
events_df["city"].value_counts().head(20)

# explore number of events per month (sorted chronologically)
events_df["month"].value_counts().sort_index()

After exploring our data, we start cleaning our data to prepare for our analysis.

In [ ]:
#  RUN ONCE

# some events have abbreviations, some have full names: for example, "Texas" and "TX" are both options
# create dictionary of state name and their state abbreviations
# we will use this to convert the state value for each event to be their state abbreviation for consistency
state_name_to_abbr = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE",
    "Florida": "FL", "Georgia": "GA", "Hawaii": "HI", "Idaho": "ID",
    "Illinois": "IL", "Indiana": "IN", "Iowa": "IA", "Kansas": "KS",
    "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS",
    "Missouri": "MO", "Montana": "MT", "Nebraska": "NE", "Nevada": "NV",
    "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM", "New York": "NY",
    "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK",
    "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
    "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT",
    "Vermont": "VT", "Virginia": "VA", "Washington": "WA", "West Virginia": "WV",
    "Wisconsin": "WI", "Wyoming": "WY", "District of Columbia": "DC",
}

# create normalize_state function to normalize state values of events
def normalize_state(val):
    # if the value is missing
    if pd.isna(val):
        return None # return None

    # convert value to a string and strip it of whitespace
    val = str(val).strip()

    # if the length of the value is already 2, it is already an abbreviation
    if len(val) == 2:
        return val.upper() # make sure to capitalize abbreviation

    # if the value is a full state name, look up its abbreviation in the dictionary
    return state_name_to_abbr.get(val, None) # return the abbreviation

# create copy of original dataframe
events_clean = events_df.copy()

# strip text columns of whitespace
events_clean["name"] = events_clean["name"].str.strip() # clean event names
events_clean["venue"] = events_clean["venue"].str.strip() # clean venues names

# only strip city if it exists
events_clean["city"] = events_clean["city"].where(
    events_clean["city"].isna(), # keep value unchanged if city is missing
    events_clean["city"].str.strip().str.title() # if there is a ciy value, remove whitespace and convert to title case
)

# apply normalize_state function to events_clean
events_clean["state"] = events_clean["state"].apply(normalize_state)

# fill missing segment, genre, subgenre categories with "Unknown"
events_clean["segment"] = events_clean["segment"].fillna("Unknown")
events_clean["genre"] = events_clean["genre"].fillna("Unknown")
events_clean["subgenre"] = events_clean["subgenre"].fillna("Unknown")

# remove ticket/products/non-events where possible, trying to preserve real events (if they have null) as much as possible
events_clean = events_clean[
    ~events_clean["name"].str.lower().str.contains(
        "voucher|package|flex|pass|seating|deposit|suite|discount|card", # if event name contains any of these words, do not keep it
        na=False
    )
]

# helper columns for dropping duplicates
events_clean["name_key"] = events_clean["name"].str.lower() # convert event name into lowercase
events_clean["venue_key"] = events_clean["venue"].str.lower() # convert event venue into lowercase

# remove duplicate rows and collapse the duplicates to just one row
events_clean = events_clean.drop_duplicates(
    subset=["name_key", "date", "venue_key"]
).drop(columns=["name_key", "venue_key"])

# reset index
events_clean = events_clean.reset_index(drop=True)

In [ ]:
#  RUN ONCE
# checking cleaned data
# NOTE: City and State columns still have unknown values, but latitude and longitude coordinates are present
# we will keep these rows since location still exists

# Compare row counts between original dataframe and cleaned dataframe
print("Original rows:", len(events_df))
print("Cleaned rows:", len(events_clean))
print("Missing values after cleaning:")
print(events_clean.isna().sum()) # count missing values per column after cleaning the dataframe

# check number of unique state values and confirm normalization of state values
print("Unique state values after cleaning:")
print(sorted(events_clean["state"].dropna().unique()))

events_clean.head(20) # print first 20 rows of cleaned dataframe

Now that the dataset has been cleaned, we will export it to a new CSV file

In [ ]:
#  RUN ONCE
output_file = "cleaned_ticketmaster_events.csv" # set name of CSV file
events_clean.to_csv(output_file, index=False) # export cleaned data to CSV file

Using our cleaned dataset of events, we will now perform our geographical data analysis.

In [ ]:
# load in cleaned CSV file
cleaned_events = pd.read_csv("cleaned_ticketmaster_events.csv")

# create SQLite database file
db = sqlite3.connect("events.db")

# convert dataframe into SQL table
cleaned_events.to_sql("events", db, if_exists="replace", index=False)

# test query to make sure SQL query works
# counts the number of events in each segment category
test_query = """
    SELECT segment, COUNT(*) as count
    FROM events
    GROUP BY segment
    ORDER BY count DESC
"""

# run the test SQL query
pd.read_sql(test_query,db)

For our analysis, we will only include events that are located in the US mainland.

In [ ]:
# counts how many events there are before filtering them by coordinates
total_before = pd.read_sql("SELECT COUNT(*) as count FROM events", db).iloc[0]["count"]

# run SQL query to only include events that are located in mainland US
cleaned_events = pd.read_sql("""
    SELECT *
    FROM events
    WHERE latitude BETWEEN 24.5 AND 49.5
    AND longitude BETWEEN -125.0 AND -66.5
""", db)

# update the sql table with the filtered data after running the sql query
cleaned_events.to_sql("events", db, index=False, if_exists="replace")

# print row counts before and after filtering events
print(f"Events before: {total_before}")
print(f"Events after: {len(cleaned_events)}")

# print filtered dataframe
cleaned_events

Now that the events have been filtered so that they are only located in mainland US, we will create a Folium map to visualize the number of events each month, categorized by their segment.

In [ ]:
# create map m centered at [38, -95] (roughly the center of the US)
m = folium.Map(location=[38, -95], zoom_start=4)

# create a dictionary of month labels, where month labels are linked to their actual month name
# we will use these month names on our map for better readability
month_labels = {
    "2026-03": "March", "2026-04": "April", "2026-05": "May",
    "2026-06": "June", "2026-07": "July", "2026-08": "August",
    "2026-09": "September", "2026-10": "October", "2026-11": "November",
    "2026-12": "December",
}

# assign colors to each segment
segment_colors = {
    "Music": "blue",
    "Sports": "red",
    "Arts & Theatre": "green",
    "Miscellaneous": "gray",
}

# create empty dictionary that will store the map layers
groups = {}

# loop through each month_labels dictionary created earlier
for month_key, label in month_labels.items():
    # create layer for each month, hide layers so that user can choose which specific months they want to see
    groups[month_key] = folium.FeatureGroup(name=label, show=False)

# loop through each row in events dataset
for _, row in cleaned_events.iterrows():
    month = row["month"] # extract month from the row
    # if the
    if month not in groups:
        continue

    color = segment_colors.get(row["segment"], "gray")

    popup_text = (
        f"<b>{row['name']}</b><br>"
        f"{row['venue']}<br>"
        f"{row['city']}, {row['state']}<br>"
        f"Date: {row['date']}<br>"
        f"Segment: {row['segment']}<br>"
        f"Genre: {row['genre']}"
    )

    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=4,
        color=color,
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(popup_text, max_width=200),
    ).add_to(groups[month])

for group in groups.values():
    group.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m.save("events_by_month.html")
m

In [ ]:
# make chorolopleth map colored by most popular genre in each state
import plotly.express as px

# find most popular genre per state, excluding vague ones
top_genre_by_state = (
    cleaned_events[~cleaned_events["genre"].isin(["Undefined", "Miscellaneous", "Other", "Unknown"])]
    .groupby(["state", "genre"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .drop_duplicates(subset="state")  # keep top genre per state
)

fig = px.choropleth(
    top_genre_by_state,
    locations="state",
    locationmode="USA-states",
    color="genre",
    scope="usa",
    hover_name="state",
    hover_data={"count": True, "genre": True},
    title="Most Popular Event Genre by State (2026)",
    color_discrete_sequence=px.colors.qualitative.Set2,
)

fig.update_layout(
    legend_title="Top Genre",
    title_font_size=16,
)

fig.show()

In [ ]:
# checking if all genres are there to confirm charts below
print("Genres in cleaned data:")
print(sorted(cleaned_events["genre"].dropna().unique()))

filtered = cleaned_events[
    ~cleaned_events["genre"].isin(["Undefined", "Miscellaneous", "Other", "Unknown"])
]

print("Genres after filtering:")
print(sorted(filtered["genre"].unique()))

print("Genres that actually win a state:")
print(sorted(top_genre_by_state["genre"].unique()))

filtered = cleaned_events[
    ~cleaned_events["genre"].isin(["Undefined","Miscellaneous","Other","Unknown"])
]

print(filtered["genre"].value_counts().head(10))

In [ ]:
# plot all categories
import matplotlib.pyplot as plt

genre_counts = (
    cleaned_events[~cleaned_events["genre"].isin(["Undefined", "Miscellaneous", "Other"])]
    .groupby(["segment", "genre"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(20)
)

# color by segment
segment_colors = {
    "Music": "steelblue",
    "Sports": "firebrick",
    "Arts & Theatre": "mediumseagreen",
    "Miscellaneous": "gray",
}
colors = genre_counts["segment"].map(segment_colors).fillna("gray")

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(
    genre_counts["genre"] + " (" + genre_counts["segment"] + ")",
    genre_counts["count"],
    color=colors
)

ax.set_xlabel("Number of Events")
ax.set_title("Top 20 Event Genres on Ticketmaster (2026)", fontsize=14)
ax.invert_yaxis()

# legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=s) for s, c in segment_colors.items()]
ax.legend(handles=legend_elements, loc="lower right")

plt.tight_layout()
plt.show()


In [ ]:
known = cleaned_events[
    ~cleaned_events["segment"].isin(["Undefined", "Miscellaneous", "Unknown"])
]

known_genres = known[
    ~known["genre"].isin(["Undefined", "Miscellaneous", "Other"])
]

# top genres using only known segments
genre_counts = (
    known_genres
    .groupby(["segment", "genre"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(20)
)

# color by segment
segment_colors = {
    "Music": "steelblue",
    "Sports": "firebrick",
    "Arts & Theatre": "mediumseagreen",
    "Miscellaneous": "gray",
}

colors = genre_counts["segment"].map(segment_colors).fillna("gray")

fig, ax = plt.subplots(figsize=(12, 7))

ax.barh(
    genre_counts["genre"] + " (" + genre_counts["segment"] + ")",
    genre_counts["count"],
    color=colors
)

ax.set_xlabel("Number of Events")
ax.set_title("Top 20 Event Genres on Ticketmaster (Known Segments Only)", fontsize=14)
ax.invert_yaxis()

legend_elements = [Patch(facecolor=c, label=s) for s, c in segment_colors.items()]
ax.legend(handles=legend_elements, loc="lower right")

plt.tight_layout()

In [ ]:
events_per_state = (
    pd.read_sql("""
        SELECT state, COUNT(*) as event_count
        FROM events
        WHERE state IS NOT NULL
        GROUP BY state
        ORDER BY event_count DESC
    """, db)
)

fig, ax = plt.subplots(figsize=(14, 7))
ax.bar(events_per_state["state"], events_per_state["event_count"], color="steelblue")
ax.set_xlabel("State")
ax.set_ylabel("Number of Events")
ax.set_title("Number of Events per State (2026)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

Below I am going to analyze the proximity of events to large US cities.

In [ ]:
# intiialize coords of large US cities
major_cities = {
    "New York":       (40.7128, -74.0060),
    "Los Angeles":    (34.0522, -118.2437),
    "Chicago":        (41.8781, -87.6298),
    "Houston":        (29.7604, -95.3698),
    "Phoenix":        (33.4484, -112.0740),
    "Philadelphia":   (39.9526, -75.1652),
    "San Antonio":    (29.4241, -98.4936),
    "San Diego":      (32.7157, -117.1611),
    "Dallas":         (32.7767, -96.7970),
    "Jacksonville":   (30.3322, -81.6557),
    "Fort Worth":     (32.7555, -97.3308),
    "San Jose":       (37.3382, -121.8863),
    "Austin":         (30.2672, -97.7431),
    "Charlotte":      (35.2271, -80.8431),
    "Columbus":       (39.9612, -82.9988),
    "Indianapolis":   (39.7684, -86.1581),
    "San Francisco":  (37.7749, -122.4194),
    "Seattle":        (47.6062, -122.3321),
    "Denver":         (39.7392, -104.9903),
    "Oklahoma City":  (35.4676, -97.5164),
    "Nashville":      (36.1627, -86.7816),
    "Washington DC":  (38.9072, -77.0369),
    "El Paso":        (31.7619, -106.4850),
    "Las Vegas":      (36.1699, -115.1398),
    "Boston":         (42.3601, -71.0589),
}

cities_df = pd.DataFrame(
    [(city, lat, lon) for city, (lat, lon) in major_cities.items()],
    columns=["city_name", "city_lat", "city_lon"]
)

In [ ]:
# calculate the distance to the nearest US city
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2): # calculates the distance from event to major US city
    # accounts for curvature of eath
    R = 3958.8  # Earth radius in miles
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

def nearest_city(lat, lon):
    best_city, best_dist = None, float("inf")
    for _, row in cities_df.iterrows(): # iterates through all cities
        d = haversine(lat, lon, row["city_lat"], row["city_lon"]) # calculates distance of event to all cities
        if d < best_dist: # if city is closer
            best_dist = d # save distance
            best_city = row["city_name"] # save name of city
    return best_city, round(best_dist, 1) # return the closest city and its distance

# apply to cleaned_events
cleaned_events[["nearest_city", "dist_to_city_miles"]] = cleaned_events.apply(
    lambda row: nearest_city(row["latitude"], row["longitude"]),
    axis=1, result_type="expand"
)

cleaned_events.to_sql("events", db, index=False, if_exists="replace")

In [ ]:
# distribution of distances
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# histogram of distances
axes[0].hist(cleaned_events["dist_to_city_miles"], bins=50, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Distance to Nearest Major City (miles)")
axes[0].set_ylabel("Number of Events")
axes[0].set_title("How Far Are Events from Major Cities?")

# % of events within thresholds
thresholds = [10, 25, 50, 100, 200]
pcts = [
    (cleaned_events["dist_to_city_miles"] <= t).mean() * 100
    for t in thresholds
]
axes[1].bar([f"≤{t} mi" for t in thresholds], pcts, color="mediumseagreen")
axes[1].set_ylabel("% of Events")
axes[1].set_title("Cumulative % of Events Within Distance of a Major City")
axes[1].set_ylim(0, 100)
for i, v in enumerate(pcts):
    axes[1].text(i, v + 1, f"{v:.1f}%", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

cleaned_events["dist_to_city_miles"].describe()

In [ ]:
from folium.plugins import HeatMap

m = folium.Map(location=[38, -95], zoom_start=4, tiles="CartoDB dark_matter")

heat_data = cleaned_events[["latitude", "longitude", "dist_to_city_miles"]].copy()
heat_data["weight"] = 1 / (heat_data["dist_to_city_miles"] + 1)

HeatMap(
    data=heat_data[["latitude", "longitude", "weight"]].values.tolist(),
    radius=12,
    blur=15,
    max_zoom=6,
    min_opacity=0.3,

).add_to(m)

# major city markers
for city, (lat, lon) in major_cities.items():
    folium.CircleMarker(
        location=[lat, lon],
        radius=2,
        color="white",
        fill=True,
        fill_color="white",
        fill_opacity=0.9,
        popup=folium.Popup(f"<b>{city}</b>", max_width=150),
    ).add_to(m)

m.save("event_proximity_heatmap.html")
m

In [ ]:
# events per nearest city
pd.read_sql("""
    SELECT nearest_city, COUNT(*) as event_count,
           ROUND(AVG(dist_to_city_miles), 1) as avg_dist
    FROM events
    GROUP BY nearest_city
    ORDER BY event_count DESC
""", db)

Ok the analysis for the proximity to major cities is between these boxes.

now we will look at prices

In [ ]:
# fetch prices
# we didnt end up using this due to limited data
all_events_price = []

months = [
    ("2026-03-01", "2026-03-31"),
    ("2026-04-01", "2026-04-30"),
    ("2026-05-01", "2026-05-31"),
    ("2026-06-01", "2026-06-30"),
    ("2026-07-01", "2026-07-31"),
    ("2026-08-01", "2026-08-31"),
    ("2026-09-01", "2026-09-30"),
    ("2026-10-01", "2026-10-31"),
    ("2026-11-01", "2026-11-30"),
    ("2026-12-01", "2026-12-31"),
]

for start, end in months:
    page = 0
    while page < 5:  # cap at 5 pages per month to stay within rate limits
        url = (
            f"https://app.ticketmaster.com/discovery/v2/events.json"
            f"?apikey={key}"
            f"&startDateTime={start}T00:00:00Z"
            f"&endDateTime={end}T23:59:59Z"
            f"&countryCode=US"
            f"&size=200"
            f"&page={page}"
        )
        response = requests.get(url)
        data = response.json()
        events = data.get("_embedded", {}).get("events", [])
        if not events:
            break

        for e in events:
            venue = e.get("_embedded", {}).get("venues", [{}])[0]
            loc = venue.get("location", {})

            # classifications
            classifications = e.get("classifications", [{}])
            primary = classifications[0] if classifications else {}
            segment  = primary.get("segment", {}).get("name")
            genre    = primary.get("genre", {}).get("name")

            # price ranges
            price_ranges = e.get("priceRanges", [])
            price_min = price_ranges[0].get("min") if price_ranges else None
            price_max = price_ranges[0].get("max") if price_ranges else None

            all_events_price.append({
                "name":      e.get("name"),
                "date":      e.get("dates", {}).get("start", {}).get("localDate"),
                "venue":     venue.get("name"),
                "city":      venue.get("city", {}).get("name"),
                "state":     venue.get("state", {}).get("stateCode"),
                "latitude":  loc.get("latitude"),
                "longitude": loc.get("longitude"),
                "month":     start[:7],
                "segment":   segment,
                "genre":     genre,
                "price_min": price_min,
                "price_max": price_max,
            })

        page_info = data.get("page", {})
        total_pages = page_info.get("totalPages", 1)
        print(f"{start[:7]} — page {page+1}/{total_pages} — {len(all_events_price)} events so far")
        if page + 1 >= total_pages:
            break
        page += 1
        time.sleep(0.25)

price_df = pd.DataFrame(all_events_price)

# clean up
price_df["state"] = price_df["state"].apply(normalize_state)
price_df = price_df.dropna(subset=["latitude", "longitude", "price_min", "price_max"])
price_df["latitude"]  = price_df["latitude"].astype(float)
price_df["longitude"] = price_df["longitude"].astype(float)

# filter to continental US
price_df = price_df[
    price_df["latitude"].between(24.5, 49.5) &
    price_df["longitude"].between(-125.0, -66.5)
]

# load into sqlite
price_df.to_sql("events_price", conn, index=False, if_exists="replace")

print(f"\nEvents with price data: {len(price_df)}")
print(f"Events WITHOUT price data: {len(pd.DataFrame(all_events_price)) - len(price_df)}")

In [ ]:
# avg price by genre
pd.read_sql("""
    SELECT genre,
           ROUND(AVG(price_min), 2) as avg_min,
           ROUND(AVG(price_max), 2) as avg_max,
           COUNT(*) as event_count
    FROM events_price
    WHERE genre NOT IN ('Undefined', 'Miscellaneous', 'Other')
    GROUP BY genre
    ORDER BY avg_min DESC
    LIMIT 15
""", conn)

In [ ]:
# avg price by state
pd.read_sql("""
    SELECT state,
           ROUND(AVG(price_min), 2) as avg_min,
           ROUND(AVG(price_max), 2) as avg_max,
           COUNT(*) as event_count
    FROM events_price
    GROUP BY state
    ORDER BY avg_min DESC
""", conn)

In [ ]:
# avg price by segment
pd.read_sql("""
    SELECT segment,
           ROUND(AVG(price_min), 2) as avg_min,
           ROUND(AVG(price_max), 2) as avg_max,
           COUNT(*) as event_count
    FROM events_price
    GROUP BY segment
    ORDER BY avg_min DESC
""", conn)

there isn't much price information, i don't know if we should use it